In [1]:

import numpy as np
import pandas as pd
import requests
cities=[{"city":"Lahore","country":"Pakistan","longitude":74.3587,"latitude":31.5204},
        {"city":"Karachi","country":"Pakistan","longitude":66.990501,"latitude":24.86096},
        {"city":"Islamabad","country":"Pakistan","longitude":73.0479,"latitude":33.6844},
        {"city":"Paris","country":"France","longitude":2.3522,"latitude":48.8566},
        {"city":"Shanghai","country":"China","longitude":121.47,"latitude":31.23},
        {"city":"Beijing","country":"China","longitude":116.4074,"latitude":39.9042},
        {"city":"Toronto","country":"Canada","longitude":-79.3832,"latitude":43.6532},
        {"city":"Calgary","country":"Canada","longitude":-114.07,"latitude":51.05},
        {"city":"Kuala Lumpur","country":"Malaysia","longitude":101.6932,"latitude":3.1409},
        {"city":"London","country":"United Kingdom","longitude":-0.1278,"latitude":51.5074},
        {"city":"Cairo","country":"Egypt","longitude":31.2357,"latitude":30.0444},
        {"city":"Sydney","country":"Australia","longitude":151.2093,"latitude":-33.8688},
        {"city":"Cape Town","country":"South Africa","longitude":18.4241,"latitude":-33.9249},
        {"city":"Jeddah","country":"Saudi Arabia","longitude":39.16,"latitude":21.53},
        {"city":"Madinah","country":"Saudi Arabia","longitude":39.6122,"latitude":24.4709},
        {"city":"Makkah","country":"Saudi Arabia","longitude":39.8262,"latitude":21.4225},
        {"city":"Bursa","country":"Turkey","longitude":29.0742,"latitude":40.1933},
        {"city":"Izmir","country":"Turkey","longitude":27.1428,"latitude":38.4237},
        {"city":"Istanbul","country":"Turkey","longitude":28.9784,"latitude":41.0082},
        {"city":"Ankara","country":"Turkey","longitude":32.8597,"latitude":39.9334}]

In [2]:

url="https://archive-api.open-meteo.com/v1/archive"
start_date="2016-01-01"
end_date="2025-12-31"
daily_variables=["temperature_2m_mean",
                 "temperature_2m_max",
                "temperature_2m_min",
                 "relative_humidity_2m_mean",
                 "dew_point_2m_mean",
                "precipitation_sum",
                 "rain_sum",
                 "precipitation_hours",
                "wind_speed_10m_max",
                 "wind_speed_10m_mean",
                 "wind_gusts_10m_max",
                "wind_direction_10m_dominant",
                 "weather_code",
                "cloud_cover_mean",
                 "pressure_msl_mean",
                 "sunshine_duration",
                 "daylight_duration",
                 "shortwave_radiation_sum"]

In [4]:
def fetch_batch(batch):
    parameters = {
        "latitude": ",".join(str(c["latitude"]) for c in batch),
        "longitude": ",".join(str(c["longitude"]) for c in batch),
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join(daily_variables),
        "models": "era5",
        "timezone": "auto",
    }
    response = requests.get(url, params=parameters)
    print(response.status_code)
    if response.status_code != 200:
        print(response.text)      # isme likha hota hai minute/hour/day mein se kaunsi limit lagi
        return None
    frames = []
    for city, result in zip(batch, response.json()):
        df = pd.DataFrame(result["daily"]).rename(columns={"time": "date"})
        df["date"] = pd.to_datetime(df["date"])
        for k in ["city", "country", "latitude", "longitude"]:
            df[k] = city[k]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

# har batch alag cell mein chalao
b1 = fetch_batch(cities[0:5]);   b1.to_csv("raw_batch1.csv", index=False)

200


In [5]:
print(b1.shape)                  # (18265, 22) honi chahiye
print(b1["city"].value_counts()) # har city ki 3653 rows
print(b1.isna().sum())           # kisi variable mein nulls to nahi
b1.head()

(18265, 23)
city
Lahore       3653
Karachi      3653
Islamabad    3653
Paris        3653
Shanghai     3653
Name: count, dtype: int64
date                           0
temperature_2m_mean            0
temperature_2m_max             0
temperature_2m_min             0
relative_humidity_2m_mean      0
dew_point_2m_mean              0
precipitation_sum              0
rain_sum                       0
precipitation_hours            0
wind_speed_10m_max             0
wind_speed_10m_mean            0
wind_gusts_10m_max             0
wind_direction_10m_dominant    0
weather_code                   0
cloud_cover_mean               0
pressure_msl_mean              0
sunshine_duration              0
daylight_duration              0
shortwave_radiation_sum        0
city                           0
country                        0
latitude                       0
longitude                      0
dtype: int64


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
0,2016-01-01,14.2,21.6,9.1,71,8.3,0.0,0.0,0.0,6.6,...,3,46,1021.0,35610.30,36478.73,10.82,Lahore,Pakistan,31.5204,74.3587
1,2016-01-02,12.1,21.1,6.5,77,7.5,0.0,0.0,0.0,10.8,...,3,37,1021.9,35453.58,36508.60,12.30,Lahore,Pakistan,31.5204,74.3587
2,2016-01-03,13.4,22.0,6.5,71,7.4,0.0,0.0,0.0,10.4,...,3,34,1018.7,35584.15,36541.12,12.20,Lahore,Pakistan,31.5204,74.3587
3,2016-01-04,14.1,19.8,8.8,77,9.8,0.6,0.6,4.0,7.8,...,51,63,1018.7,28057.65,36576.21,9.30,Lahore,Pakistan,31.5204,74.3587
4,2016-01-05,14.8,22.1,9.6,79,10.8,0.0,0.0,0.0,9.6,...,3,27,1018.6,35564.07,36613.81,12.77,Lahore,Pakistan,31.5204,74.3587


In [6]:
b2 = fetch_batch(cities[5:10]);   b2.to_csv("raw_batch2.csv", index=False)

200


In [9]:
print(b2.shape)                  # (18265, 22) honi chahiye
print(b2["city"].value_counts()) # har city ki 3653 rows
print(b2.isna().sum())           # kisi variable mein nulls to nahi
display(b2.head())
b2.tail()

(18265, 23)
city
Beijing         3653
Toronto         3653
Calgary         3653
Kuala Lumpur    3653
London          3653
Name: count, dtype: int64
date                           0
temperature_2m_mean            0
temperature_2m_max             0
temperature_2m_min             0
relative_humidity_2m_mean      0
dew_point_2m_mean              0
precipitation_sum              0
rain_sum                       0
precipitation_hours            0
wind_speed_10m_max             0
wind_speed_10m_mean            0
wind_gusts_10m_max             0
wind_direction_10m_dominant    0
weather_code                   0
cloud_cover_mean               0
pressure_msl_mean              0
sunshine_duration              0
daylight_duration              0
shortwave_radiation_sum        0
city                           0
country                        0
latitude                       0
longitude                      0
dtype: int64


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
0,2016-01-01,-1.9,4.1,-6.7,69,-7.1,0.0,0.0,0.0,9.8,...,0,1,1023.7,32016.81,33768.89,8.85,Beijing,China,39.9042,116.4074
1,2016-01-02,-0.1,7.9,-6.3,65,-6.6,0.0,0.0,0.0,6.6,...,1,10,1020.2,32067.18,33810.65,8.89,Beijing,China,39.9042,116.4074
2,2016-01-03,-1.4,3.1,-4.3,77,-5.1,0.0,0.0,0.0,9.9,...,2,10,1023.0,32119.29,33856.13,8.87,Beijing,China,39.9042,116.4074
3,2016-01-04,-0.9,2.8,-4.0,37,-14.3,0.0,0.0,0.0,12.4,...,3,30,1030.0,32173.01,33905.26,8.15,Beijing,China,39.9042,116.4074
4,2016-01-05,-3.2,1.7,-6.9,36,-16.8,0.0,0.0,0.0,8.7,...,0,0,1030.3,32228.22,33957.91,9.50,Beijing,China,39.9042,116.4074


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
18260,2025-12-27,5.6,7.8,2.8,87,3.5,0.0,0.0,0.0,28.1,...,3,52,1034.0,25380.43,28297.12,3.23,London,United Kingdom,51.5074,-0.1278
18261,2025-12-28,6.7,7.5,4.8,80,3.4,0.0,0.0,0.0,19.1,...,3,90,1032.3,1733.33,28337.14,1.46,London,United Kingdom,51.5074,-0.1278
18262,2025-12-29,5.4,6.3,4.2,81,2.4,0.0,0.0,0.0,15.6,...,3,74,1029.2,3751.46,28383.28,1.79,London,United Kingdom,51.5074,-0.1278
18263,2025-12-30,3.9,6.4,0.7,86,1.7,0.0,0.0,0.0,15.2,...,3,40,1033.5,20309.04,28435.48,2.84,London,United Kingdom,51.5074,-0.1278
18264,2025-12-31,0.7,3.4,-0.7,92,-0.5,0.0,0.0,0.0,15.9,...,1,5,1028.9,27684.15,28493.72,3.88,London,United Kingdom,51.5074,-0.1278


In [10]:
b3 = fetch_batch(cities[10:15]);  b3.to_csv("raw_batch3.csv", index=False)

200


In [11]:
print(b3.shape)                  # (18265, 22) honi chahiye
print(b3["city"].value_counts()) # har city ki 3653 rows
print(b3.isna().sum())           # kisi variable mein nulls to nahi
display(b3.head())
b3.tail()

(18265, 23)
city
Cairo        3653
Sydney       3653
Cape Town    3653
Jeddah       3653
Madinah      3653
Name: count, dtype: int64
date                           0
temperature_2m_mean            0
temperature_2m_max             0
temperature_2m_min             0
relative_humidity_2m_mean      0
dew_point_2m_mean              0
precipitation_sum              0
rain_sum                       0
precipitation_hours            0
wind_speed_10m_max             0
wind_speed_10m_mean            0
wind_gusts_10m_max             0
wind_direction_10m_dominant    0
weather_code                   0
cloud_cover_mean               0
pressure_msl_mean              0
sunshine_duration              0
daylight_duration              0
shortwave_radiation_sum        0
city                           0
country                        0
latitude                       0
longitude                      0
dtype: int64


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
0,2016-01-01,12.5,14.7,9.2,66,5.8,0.7,0.7,4.0,25.5,...,51,48,1021.8,36000.00,36900.35,11.64,Cairo,Egypt,30.0444,31.2357
1,2016-01-02,11.2,15.7,8.3,62,3.9,0.0,0.0,0.0,15.3,...,3,52,1027.0,18581.76,36928.69,9.06,Cairo,Egypt,30.0444,31.2357
2,2016-01-03,12.7,18.7,8.0,61,5.0,0.0,0.0,0.0,18.2,...,3,36,1024.8,35897.15,36959.50,11.32,Cairo,Egypt,30.0444,31.2357
3,2016-01-04,14.7,20.7,9.8,53,4.7,0.0,0.0,0.0,20.6,...,2,16,1020.9,36000.00,36992.73,13.53,Cairo,Egypt,30.0444,31.2357
4,2016-01-05,15.8,22.0,10.6,40,1.7,0.0,0.0,0.0,21.1,...,0,0,1019.4,36000.00,37028.31,13.91,Cairo,Egypt,30.0444,31.2357


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
18260,2025-12-27,19.7,26.6,14.4,60,9.8,0.0,0.0,0.0,21.2,...,3,23,1011.5,37102.56,38255.19,15.57,Madinah,Saudi Arabia,24.4709,39.6122
18261,2025-12-28,18.1,24.9,10.6,48,5.4,0.0,0.0,0.0,20.6,...,0,1,1012.3,37793.37,38267.51,16.77,Madinah,Saudi Arabia,24.4709,39.6122
18262,2025-12-29,17.8,24.8,10.9,48,4.7,0.0,0.0,0.0,18.8,...,0,0,1015.7,37807.71,38281.75,16.56,Madinah,Saudi Arabia,24.4709,39.6122
18263,2025-12-30,17.6,23.1,12.4,41,2.8,0.0,0.0,0.0,10.0,...,0,0,1019.5,37823.97,38297.92,16.85,Madinah,Saudi Arabia,24.4709,39.6122
18264,2025-12-31,18.3,26.9,10.0,26,-2.7,0.0,0.0,0.0,15.9,...,0,0,1016.7,37842.13,38316.01,17.12,Madinah,Saudi Arabia,24.4709,39.6122


In [18]:
b4 = fetch_batch(cities[15:20])

200


In [19]:
print(b4.shape)                  # (18265, 22) honi chahiye
print(b4["city"].value_counts()) # har city ki 3653 rows
print(b4.isna().sum())           # kisi variable mein nulls to nahi
display(b4.head())
b4.tail()

(18265, 23)
city
Makkah      3653
Bursa       3653
Izmir       3653
Istanbul    3653
Ankara      3653
Name: count, dtype: int64
date                           0
temperature_2m_mean            0
temperature_2m_max             0
temperature_2m_min             0
relative_humidity_2m_mean      0
dew_point_2m_mean              0
precipitation_sum              0
rain_sum                       0
precipitation_hours            0
wind_speed_10m_max             0
wind_speed_10m_mean            0
wind_gusts_10m_max             0
wind_direction_10m_dominant    0
weather_code                   0
cloud_cover_mean               0
pressure_msl_mean              0
sunshine_duration              0
daylight_duration              0
shortwave_radiation_sum        0
city                           0
country                        0
latitude                       0
longitude                      0
dtype: int64


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
0,2016-01-01,21.2,25.3,17.7,63,13.6,0.0,0.0,0.0,16.5,...,3,75,1016.5,38587.28,39047.61,12.29,Makkah,Saudi Arabia,21.4225,39.8262
1,2016-01-02,19.8,24.1,15.8,56,10.2,0.0,0.0,0.0,19.3,...,1,5,1019.7,38606.57,39066.52,17.37,Makkah,Saudi Arabia,21.4225,39.8262
2,2016-01-03,19.9,27.7,14.5,53,9.3,0.0,0.0,0.0,15.6,...,0,0,1018.0,38627.49,39087.08,18.12,Makkah,Saudi Arabia,21.4225,39.8262
3,2016-01-04,21.8,26.4,17.4,61,13.8,0.0,0.0,0.0,16.1,...,3,71,1016.1,15395.69,39109.26,8.13,Makkah,Saudi Arabia,21.4225,39.8262
4,2016-01-05,22.6,27.5,18.0,63,15.0,0.0,0.0,0.0,10.9,...,3,56,1016.8,38583.65,39133.01,13.71,Makkah,Saudi Arabia,21.4225,39.8262


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,dew_point_2m_mean,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,weather_code,cloud_cover_mean,pressure_msl_mean,sunshine_duration,daylight_duration,shortwave_radiation_sum,city,country,latitude,longitude
18260,2025-12-27,1.5,3.5,-1.0,78,-2.0,1.0,1.0,9.0,17.1,...,51,57,1014.7,29962.52,33640.38,8.46,Ankara,Turkey,39.9334,32.8597
18261,2025-12-28,-0.0,2.0,-1.9,85,-2.3,2.1,1.6,13.0,21.4,...,71,85,1010.4,24464.20,33664.22,6.02,Ankara,Turkey,39.9334,32.8597
18262,2025-12-29,-0.9,1.1,-2.4,81,-3.9,0.5,0.4,4.0,24.6,...,71,59,1014.0,31409.87,33691.80,8.03,Ankara,Turkey,39.9334,32.8597
18263,2025-12-30,0.7,5.9,-3.7,68,-5.0,0.4,0.4,1.0,15.5,...,51,43,1015.9,31463.90,33723.08,9.01,Ankara,Turkey,39.9334,32.8597
18264,2025-12-31,1.5,3.9,-2.5,84,-0.9,4.9,4.9,12.0,14.7,...,53,78,1008.1,21794.57,33758.07,5.53,Ankara,Turkey,39.9334,32.8597
